# RetailMind AI — Phase 4: Demand Forecasting

**Phase:** 4 — Demand Forecasting  
**Stage:** 1 — Data Loading and Verification

---

## Objective

Develop demand forecasting models using the engineered daily product demand dataset produced in Phase 3 — Feature Engineering.

| Property | Value |
|---|---|
| **Target variable** | `DailyQuantity` — total units sold for a product on a given date |
| **Granularity** | `ProductID` × `Date` |
| **Evaluation metrics** | MAE, RMSE, MAPE *(implemented in a later stage)* |
| **Split strategy** | Time-based train / validation / test split *(implemented in a later stage)* |

---

## Stage 1 Scope

This stage covers **data loading and verification only**.

The following steps are deferred to later stages:
- Feature selection for forecasting
- Handling lag/rolling NaN values appropriately
- Time-based train/validation/test split
- Model training and evaluation

**Prerequisites:**
- `01_data_understanding.ipynb` — Data quality validation ✅
- `02_eda.ipynb` — Exploratory analysis ✅
- `03_feature_engineering.ipynb` — Feature engineering ✅

**Input dataset:**
`data/processed/daily_product_demand.csv`

---
## Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded successfully.')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

Libraries loaded successfully.
  pandas  : 3.0.5
  numpy   : 2.4.0


---
## Cell 2 — Dataset Path

In [2]:
DEMAND_PATH = '../data/processed/daily_product_demand.csv'

# Confirm file exists before loading
if os.path.exists(DEMAND_PATH):
    size_kb = os.path.getsize(DEMAND_PATH) / 1024
    print(f'Dataset found: {os.path.abspath(DEMAND_PATH)}')
    print(f'File size    : {size_kb:,.1f} KB')
else:
    raise FileNotFoundError(
        f'STOPPING: Dataset not found at {DEMAND_PATH}. '
        'Run 03_feature_engineering.ipynb first.'
    )

Dataset found: D:\RetailMindAI\RetailMindAI\data\processed\daily_product_demand.csv
File size    : 43,391.3 KB


---
## Cell 3 — Load Dataset

> The dataset is loaded **as-is** from Phase 3. No rows are dropped, no NaN values are filled, and no transformations are applied at this stage.

In [3]:
demand = pd.read_csv(DEMAND_PATH)

# Convert Date to datetime immediately after loading
demand['Date'] = pd.to_datetime(demand['Date'])

print('Dataset loaded successfully.')
print(f'Shape: {demand.shape[0]:,} rows x {demand.shape[1]} columns')

Dataset loaded successfully.
Shape: 91,250 rows x 48 columns


---
## Cell 4 — Dataset Verification

Print key characteristics of the loaded dataset to confirm it matches the expected Phase 3 output.

In [4]:
print('=' * 55)
print('  DATASET VERIFICATION')
print('=' * 55)

# Rows and columns
print(f'  Rows               : {demand.shape[0]:,}')
print(f'  Columns            : {demand.shape[1]}')

# Date range
date_min = demand['Date'].min()
date_max = demand['Date'].max()
n_dates  = demand['Date'].nunique()
print(f'  Date range         : {date_min.date()} to {date_max.date()}')
print(f'  Unique dates       : {n_dates:,}')

# Products
n_products = demand['ProductID'].nunique()
print(f'  Unique products    : {n_products}')

# Target column
target_col = 'DailyQuantity'
target_present = target_col in demand.columns
print(f'  Target column      : {target_col}  (present: {target_present})')

# Missing values in target
target_missing = demand[target_col].isna().sum()
print(f'  Missing in target  : {target_missing}')

print('=' * 55)

  DATASET VERIFICATION
  Rows               : 91,250
  Columns            : 48
  Date range         : 2020-01-01 to 2024-12-29
  Unique dates       : 1,825
  Unique products    : 50
  Target column      : DailyQuantity  (present: True)
  Missing in target  : 0


---
## Cell 5 — Column Overview

In [5]:
print(f'All columns ({demand.shape[1]} total):')
for i, col in enumerate(demand.columns, 1):
    dtype   = str(demand[col].dtype)
    n_miss  = demand[col].isna().sum()
    miss_pct = n_miss / len(demand) * 100
    marker  = '  <- TARGET' if col == 'DailyQuantity' else ''
    print(f'  {i:2d}. {col:<35} dtype={dtype:<10} missing={n_miss:,} ({miss_pct:.2f}%){marker}')

All columns (48 total):
   1. ProductID                           dtype=str        missing=0 (0.00%)
   2. Date                                dtype=datetime64[us] missing=0 (0.00%)
   3. DailyQuantity                       dtype=float64    missing=0 (0.00%)  <- TARGET
   4. DailyRevenue                        dtype=float64    missing=0 (0.00%)
   5. DailyOrderCount                     dtype=float64    missing=0 (0.00%)
   6. DailyAveragePrice                   dtype=float64    missing=0 (0.00%)
   7. DailyAverageDiscount                dtype=float64    missing=0 (0.00%)
   8. DailyReturnCount                    dtype=float64    missing=0 (0.00%)
   9. DailyCancellationCount              dtype=float64    missing=0 (0.00%)
  10. DailyAverageOrderValue              dtype=float64    missing=0 (0.00%)
  11. ProductName                         dtype=str        missing=0 (0.00%)
  12. year                                dtype=int64      missing=0 (0.00%)
  13. month                          

---
## Cell 6 — Sample Rows

Display a small sample to visually confirm the dataset structure.

In [6]:
display_cols = [
    'ProductID', 'Date', 'DailyQuantity',
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28',
    'is_weekend', 'month_number', 'year'
]
print('Sample: first 10 rows of product P00001')
sample = demand[demand['ProductID'] == demand['ProductID'].iloc[0]][display_cols].head(10)
display(sample)

Sample: first 10 rows of product P00001


,ProductID,Date,DailyQuantity,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,is_weekend,month_number,year
0,P00001,2020-01-01,2.0000,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2020
1,P00001,2020-01-02,0.0000,2.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
2,P00001,2020-01-03,5.0000,0.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
3,P00001,2020-01-04,3.0000,5.0000,NaN,NaN,NaN,NaN,NaN,1,1,2020
4,P00001,2020-01-05,1.0000,3.0000,NaN,NaN,NaN,NaN,NaN,1,1,2020
5,P00001,2020-01-06,2.0000,1.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
6,P00001,2020-01-07,0.0000,2.0000,NaN,NaN,NaN,NaN,NaN,0,1,2020
7,P00001,2020-01-08,7.0000,0.0000,2.0000,NaN,NaN,1.8571,NaN,0,1,2020
8,P00001,2020-01-09,3.0000,7.0000,0.0000,NaN,NaN,2.5714,NaN,0,1,2020
9,P00001,2020-01-10,5.0000,3.0000,5.0000,NaN,NaN,3.0000,NaN,0,1,2020


---
## Cell 7 — Lag / Rolling NaN Summary

> **Important:** The NaN values shown below in lag and rolling features are **expected and correct**. They arise because the first N days of each product's history cannot have a valid N-day lookback.

**These NaN values must NOT be filled or dropped at this stage.** They will be handled appropriately during feature selection and train/test split preparation in the next stage.

In [7]:
lag_rolling_cols = [
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_std_7',
    'rolling_mean_14', 'rolling_std_14',
    'rolling_mean_28', 'rolling_std_28',
    'short_term_mean', 'medium_term_mean', 'long_term_mean',
    'short_vs_medium_growth', 'medium_vs_long_growth',
    'cv_7', 'cv_28',
    'rolling_return_rate_28', 'rolling_cancel_rate_28',
    'rolling_revenue_mean_28', 'rolling_revenue_std_28',
    'revenue_growth_7_vs_28'
]

n_products = demand['ProductID'].nunique()
print('Lag/Rolling NaN Summary (all expected):')
print(f'  (Based on {n_products} products with complete daily grid)')
print()
for col in lag_rolling_cols:
    if col in demand.columns:
        n_nan = demand[col].isna().sum()
        pct   = n_nan / len(demand) * 100
        print(f'  {col:<30} : {n_nan:,} NaN ({pct:.2f}%)')
print()
print('NOTE: Do NOT fill or drop these NaN values at this stage.')

Lag/Rolling NaN Summary (all expected):
  (Based on 50 products with complete daily grid)

  lag_1                          : 50 NaN (0.05%)
  lag_7                          : 350 NaN (0.38%)
  lag_14                         : 700 NaN (0.77%)
  lag_28                         : 1,400 NaN (1.53%)
  rolling_mean_7                 : 350 NaN (0.38%)
  rolling_std_7                  : 350 NaN (0.38%)
  rolling_mean_14                : 700 NaN (0.77%)
  rolling_std_14                 : 700 NaN (0.77%)
  rolling_mean_28                : 1,400 NaN (1.53%)
  rolling_std_28                 : 1,400 NaN (1.53%)
  short_term_mean                : 350 NaN (0.38%)
  medium_term_mean               : 1,400 NaN (1.53%)
  long_term_mean                 : 4,500 NaN (4.93%)
  short_vs_medium_growth         : 1,400 NaN (1.53%)
  medium_vs_long_growth          : 4,500 NaN (4.93%)
  cv_7                           : 396 NaN (0.43%)
  cv_28                          : 1,400 NaN (1.53%)
  rolling_return_rate_28   

---
## Cell 8 — Expected Phase 3 Characteristics Check

Assert that the dataset matches the documented Phase 3 output exactly.

In [8]:
print('=' * 55)
print('  PHASE 3 CHARACTERISTICS CHECK')
print('=' * 55)

checks = []

# Check 1: row count
actual_rows = demand.shape[0]
row_ok = (actual_rows == 91250)
checks.append(('Rows == 91,250', row_ok, f'actual={actual_rows:,}'))

# Check 2: unique products
actual_prods = demand['ProductID'].nunique()
prod_ok = (actual_prods == 50)
checks.append(('Unique products == 50', prod_ok, f'actual={actual_prods}'))

# Check 3: start date
actual_start = demand['Date'].min().date()
from datetime import date
start_ok = (actual_start == date(2020, 1, 1))
checks.append(('Start date == 2020-01-01', start_ok, f'actual={actual_start}'))

# Check 4: end date
actual_end = demand['Date'].max().date()
end_ok = (actual_end == date(2024, 12, 29))
checks.append(('End date == 2024-12-29', end_ok, f'actual={actual_end}'))

# Check 5: DailyQuantity present
target_ok = ('DailyQuantity' in demand.columns)
checks.append(('DailyQuantity column exists', target_ok, ''))

# Check 6: DailyQuantity has no missing values
target_miss = demand['DailyQuantity'].isna().sum()
target_miss_ok = (target_miss == 0)
checks.append(('DailyQuantity has 0 missing', target_miss_ok, f'actual={target_miss}'))

# Check 7: no duplicate ProductID-Date pairs
n_dup = demand.duplicated(subset=['ProductID', 'Date']).sum()
dup_ok = (n_dup == 0)
checks.append(('No duplicate ProductID-Date rows', dup_ok, f'duplicates={n_dup}'))

all_passed = True
for label, passed, detail in checks:
    status = 'PASS' if passed else 'FAIL'
    detail_str = f'  ({detail})' if detail else ''
    print(f'  [{status}] {label}{detail_str}')
    if not passed:
        all_passed = False

print('=' * 55)
print(f"  OVERALL: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED'}")
print('=' * 55)

  PHASE 3 CHARACTERISTICS CHECK
  [PASS] Rows == 91,250  (actual=91,250)
  [PASS] Unique products == 50  (actual=50)
  [PASS] Start date == 2020-01-01  (actual=2020-01-01)
  [PASS] End date == 2024-12-29  (actual=2024-12-29)
  [PASS] DailyQuantity column exists
  [PASS] DailyQuantity has 0 missing  (actual=0)
  [PASS] No duplicate ProductID-Date rows  (duplicates=0)
  OVERALL: ALL CHECKS PASSED


---
## Cell 9 — Stage 1 Summary

In [9]:
print('=' * 55)
print('  STAGE 1 COMPLETE — Data Loading & Verification')
print('=' * 55)

print('  Dataset loaded : data/processed/daily_product_demand.csv')
print(f'  Rows           : {demand.shape[0]:,}')
print(f'  Columns        : {demand.shape[1]}')
print(f'  Products       : {demand["ProductID"].nunique()}')
print(f'  Date range     : {demand["Date"].min().date()} to {demand["Date"].max().date()}')
print(f'  Target         : DailyQuantity  (0 missing values)')
print(f'  Dataset status : READY for Phase 4 Stage 2')

print()
print('  NEXT STAGE (implement separately):')
print('  - Select forecasting features from the 48 available columns')
print('  - Handle lag/rolling NaN values (drop or mask for training)')
print('  - Apply time-based train/validation/test split')
print('  - Train demand forecasting model')
print('  - Evaluate with MAE, RMSE, MAPE')

print('=' * 55)
print('  Do NOT proceed into model training in this notebook.')
print('=' * 55)

  STAGE 1 COMPLETE — Data Loading & Verification
  Dataset loaded : data/processed/daily_product_demand.csv
  Rows           : 91,250
  Columns        : 48
  Products       : 50
  Date range     : 2020-01-01 to 2024-12-29
  Target         : DailyQuantity  (0 missing values)
  Dataset status : READY for Phase 4 Stage 2

  NEXT STAGE (implement separately):
  - Select forecasting features from the 48 available columns
  - Handle lag/rolling NaN values (drop or mask for training)
  - Apply time-based train/validation/test split
  - Train demand forecasting model
  - Evaluate with MAE, RMSE, MAPE
  Do NOT proceed into model training in this notebook.
